### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [1]:
from langchain_classic.document_loaders import TextLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [2]:
## Step 1: Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [3]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [4]:
## Step 2: Vector Store
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)

## Step 3: MMR Retriever
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k":5})
retriever

d:\Mission Project\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2467.76it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000016229A75A90>, search_type='mmr', search_kwargs={'k': 5})

In [5]:
## Step 4: LLM and Prompt
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]= os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000162350E06E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000162350E1BE0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
## Query Expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrival by adding relevant synonyms, technical terms, and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrival by adding relevant synonyms, technical terms, and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000162350E06E0>, async_client=<groq.resources.chat.completions.AsyncCompletion

In [7]:
query_expansion_chain.invoke({"query": "Langchain memory"})

'**Expanded query**\n\n```\n"LangChain memory" OR "LangChain state management" OR "LangChain conversation memory" OR "LangChain chat history" OR \n"LangChain memory backends" OR "LangChain memory providers" OR "LangChain memory modules" OR "LangChain memory buffer" OR \n"LangChain memory store" OR "LangChain memory persistence" OR "LangChain memory examples" OR \n"ConversationBufferMemory" OR "ConversationSummaryMemory" OR "ConversationTokenBufferMemory" OR \n"ChatMessageHistory" OR "RedisChatMessageHistory" OR "PostgresChatMessageHistory" OR "MongoDBChatMessageHistory" OR \n"VectorStoreRetrieverMemory" OR "FAISS memory store" OR "Chroma memory store" OR "Milvus memory store" OR "Pinecone memory store" OR \n"LLMChain memory handling" OR "Agent memory in LangChain" OR "MemoryManager" OR "MemoryCache" OR \n"persistent memory for LangChain agents" OR "in‑memory vs persistent LangChain memory" OR \n"multi‑turn dialogue memory LangChain" OR "LangChain memory architecture" OR "LangChain memo

In [8]:
## RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm=llm, prompt=answer_prompt)

In [9]:
## Step 5: Full RAG Pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [10]:
## Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print(" Answer:\n", response)

**Expanded query**

```
What memory types and back‑ends does LangChain support?  (e.g. built‑in memory classes such as ConversationBufferMemory, ConversationSummaryMemory, ConversationBufferWindowMemory, SummaryMemory, EntityMemory, VectorStoreRetrieverMemory, etc., as well as persistent or external options like InMemory, RedisCache, SQLite, FAISS, Chroma, Milvus, Pinecone, Weaviate, Azure Cognitive Search, etc.)  Include synonyms such as “memory back‑ends”, “stateful memory mechanisms”, “memory modules”, “memory implementations”, and any related technical terminology used in the LangChain documentation. 
```
 Answer:
 LangChain provides built‑in memory modules such as:

- **ConversationBufferMemory** – stores the full dialogue history so the model can reference previous turns.  
- **ConversationSummaryMemory** – keeps a summarized version of the conversation, allowing the model to stay aware of long interactions without exceeding token limits.


In [11]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

**Expanded query**

```
CrewAI agents OR Crew AI agents OR CrewAI autonomous agents OR CrewAI intelligent agents OR CrewAI software agents OR CrewAI digital assistants OR CrewAI bots OR CrewAI virtual agents OR CrewAI multi‑agent system OR CrewAI platform OR CrewAI SDK OR CrewAI API OR CrewAI documentation OR CrewAI tutorial OR CrewAI examples OR CrewAI use cases OR CrewAI implementation guide OR CrewAI Python library OR CrewAI GitHub repository OR CrewAI LLM‑based agents OR CrewAI prompt engineering OR CrewAI task decomposition OR CrewAI agentic workflow OR CrewAI multi‑agent orchestration OR AI crew management agents OR autonomous AI agents for crew coordination OR intelligent crew assistants OR AI‑powered crew automation OR multi‑agent coordination framework OR large language model agents OR task‑oriented AI agents OR AI‑driven crew scheduling OR AI‑assisted team collaboration OR agentic workflow automation
```
✅ Answer:
 **CrewAI agents are autonomous, role‑based AI components that